# INDILEX Long-Judgment Annotation Pipeline

**Architecture:** Three-stage local-model pipeline for long Indian legal judgments.

```
LONG JUDGMENT
    ↓  safe normalization
    ↓  token-aware chunking
    ↓  chunk-level structured evidence extraction   (Stage A)
    ↓  cross-chunk evidence aggregation             (Stage B)
    ↓  case-conditioned final annotation            (Stage C)
    ↓  validation + checkpoint
FINAL CSV
```

**Phase 1 — Preprocessing (no model, run once):**
Load → validate → normalize → split by case-type into batch CSVs →
tokenize → compute chunks → save chunk JSONL.

**Phase 2 — Annotation (vLLM model, resumable):**
Load model → for each judgment: evidence extraction → aggregation →
final annotation → validate → checkpoint → final CSV.

**Testing sequence:**
1. `DRY_RUN = True` — verify prompts, chunking, routing (zero model calls)
2. `MAX_JUDGMENTS = 1` — one real judgment end-to-end
3. `MAX_JUDGMENTS = 5` — small batch
4. `MAX_JUDGMENTS = None` — full run

**Model:** Qwen2.5-14B-Instruct via vLLM (tensor-parallel across 2× GPUs)


In [ ]:
!du -sh /mnt/Data/yashv7523/.cache/huggingface/

In [ ]:
!df -h /mnt/Data

In [ ]:
!rm -rf /mnt/Data/yashv7523/.cache/huggingface/hub/*

In [ ]:
!du -sh /mnt/Data/yashv7523/.cache/huggingface/

In [ ]:
!df -h

In [ ]:
# 1. Sabse pehle requirements install karein (Sirf pehli baar run karna hai)
!pip install tiktoken --only-binary:all:
!pip install vllm==0.6.1.post1 transformers pandas tqdm

# 2. Kernel ko RESTART karein iske baad (Zaroori hai!)

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


In [1]:
import os
import multiprocessing as mp

# vLLM aur PyTorch ko force karein sahi method use karne ke liye
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["MASTER_PORT"] = "29560"  # Har baar port badalna safe rehta hai

try:
    mp.set_start_method('spawn', force=True)
    print("Spawn method and clean ports successfully set!")
except RuntimeError as e:
    print(f"Error setting start method: {e}")

Spawn method and clean ports successfully set!


## cell1 import cell 

In [2]:
import os, sys, json, re, ast, hashlib, time, logging, copy, shutil, gc
import unicodedata, warnings, traceback
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict, Counter
from typing import Optional, Dict, List, Any, Tuple

import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)


/mnt/Data/yashv7523/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/mnt/Data/yashv7523/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## ════════════════════════════════════════════════════════════════
# cell 2 OPERATOR CONFIGURATION — edit this cell only
# ════════════════════════════════════════════════════════════════

In [3]:
# ════════════════════════════════════════════════════════════════
# OPERATOR CONFIGURATION — edit this cell only
# ════════════════════════════════════════════════════════════════

# ── Input / Output ──────────────────────────────────────────────
INPUT_FILE   = "INDILEX_chunk_candidates.csv"       # ← EDIT
OUTPUT_BASE_DIR = "Chunk_output"                # ← EDIT

# ── Model ───────────────────────────────────────────────────────
MODEL_NAME_OR_PATH = "Qwen/Qwen2.5-7B-Instruct"
BACKEND = "vllm"                       # "vllm" (recommended) or "transformers"
TENSOR_PARALLEL_SIZE = 2               # number of GPUs for vLLM tensor parallelism
GPU_MEMORY_UTILIZATION = 0.90          # fraction of GPU VRAM vLLM may use

# ── Chunking ────────────────────────────────────────────────────
CHUNK_SIZE_TOKENS   = 7000
CHUNK_OVERLAP_TOKENS = 500

# ── Generation ──────────────────────────────────────────────────
MAX_NEW_TOKENS_EVIDENCE    = 2048      # per-chunk evidence extraction
MAX_NEW_TOKENS_AGGREGATION = 3072      # evidence aggregation
MAX_NEW_TOKENS_FINAL       = 1024      # final 6-field annotation
TEMPERATURE = 0.0                      # 0 = deterministic greedy decoding

# ── Batching ────────────────────────────────────────────────────
BATCH_SIZE = 20

# ── Robustness ──────────────────────────────────────────────────
MAX_RETRIES         = 3
RETRY_FAILED_ON_RESUME = True

# ── Run control ─────────────────────────────────────────────────
DRY_RUN        = False                # True = show prompts, zero model calls
MAX_JUDGMENTS  = 1                  # None = all; int = limit per batch
PROCESS_BATCHES = None                 # None = all batches; or ["criminal_batch_001", ...]
FORCE_REPROCESS_PHASE1 = False         # True = re-create batches/chunks even if they exist

# ── Misc ────────────────────────────────────────────────────────
DEBUG = False

# ── Derived (do not edit) ───────────────────────────────────────
INPUT_PATH = Path(INPUT_FILE)
OUTPUT_DIR = Path(OUTPUT_BASE_DIR)
assert INPUT_PATH.suffix.lower() == ".csv", "INPUT_FILE must be a .csv"


In [4]:
# ── Directory structure ──────────────────────────────────────────
BATCHES_DIR           = OUTPUT_DIR / "batches"
CHUNKS_DIR            = OUTPUT_DIR / "chunks"
CHUNK_EVIDENCE_DIR    = OUTPUT_DIR / "chunk_evidence"
AGGREGATED_EVIDENCE_DIR = OUTPUT_DIR / "aggregated_evidence"
CHECKPOINTS_DIR       = OUTPUT_DIR / "checkpoints"
ERRORS_DIR            = OUTPUT_DIR / "errors"
LOGS_DIR              = OUTPUT_DIR / "logs"
FINAL_DIR             = OUTPUT_DIR / "final"

for d in [BATCHES_DIR, CHUNKS_DIR, CHUNK_EVIDENCE_DIR,
          AGGREGATED_EVIDENCE_DIR, CHECKPOINTS_DIR, ERRORS_DIR,
          LOGS_DIR, FINAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Logging ─────────────────────────────────────────────────────
LOG_PATH = LOGS_DIR / "indilex_long_pipeline.log"
logger = logging.getLogger("indilex")
logger.setLevel(logging.DEBUG if DEBUG else logging.INFO)
logger.handlers.clear()
_fh = logging.FileHandler(LOG_PATH, encoding="utf-8")
_fh.setLevel(logging.DEBUG if DEBUG else logging.INFO)
_fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s"))
_sh = logging.StreamHandler(sys.stderr)
_sh.setLevel(logging.WARNING)
_sh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s"))
logger.addHandler(_fh)
logger.addHandler(_sh)
logger.info("Pipeline started | output_dir=%s | model=%s | dry_run=%s",
            OUTPUT_DIR, MODEL_NAME_OR_PATH, DRY_RUN)

print("Output directory :", OUTPUT_DIR)
print("Log file         :", LOG_PATH)


Output directory : Chunk_output
Log file         : Chunk_output/logs/indilex_long_pipeline.log


In [5]:
# ── Helpers ──────────────────────────────────────────────────────

def is_missing(val) -> bool:
    """True for None, NaN, empty/whitespace, or sentinel strings."""
    if val is None:
        return True
    if isinstance(val, float):
        import math
        return math.isnan(val)
    s = str(val).strip().lower()
    return s in ("", "nan", "none", "null", "na", "n/a")

def safe_str(val) -> str:
    if val is None:
        return ""
    if isinstance(val, float):
        import math
        if math.isnan(val):
            return ""
    return str(val)

def now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

CANONICAL_CASE_TYPES = ["Criminal", "Civil", "Constitutional", "Administrative"]

def normalize_case_type(raw) -> Optional[str]:
    """Normalize case-type string to canonical form."""
    if raw is None:
        return None
    s = str(raw).strip().lower()
    if not s or s in ("nan", "none", "null"):
        return None
    _map = {"crim": "Criminal", "civ": "Civil", "const": "Constitutional",
            "consti": "Constitutional", "admin": "Administrative"}
    for prefix, canon in _map.items():
        if s.startswith(prefix):
            return canon
    return None

def judgment_hash(text) -> str:
    """MD5 of whitespace-collapsed lowercase text for dedup."""
    if is_missing(text):
        return ""
    normalized = re.sub(r"\s+", " ", str(text).strip().lower())
    return hashlib.md5(normalized.encode("utf-8")).hexdigest()

def atomic_save_csv(df: pd.DataFrame, path: Path) -> None:
    tmp = path.with_suffix(".tmp")
    df.to_csv(tmp, index=False, encoding="utf-8-sig")
    os.replace(str(tmp), str(path))

def atomic_write_json(data: Any, path: Path) -> None:
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(str(tmp), str(path))

def append_jsonl(record: dict, path: Path) -> None:
    """Append one JSON record to a JSONL file (crash-safe append)."""
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def read_jsonl(path: Path) -> List[dict]:
    """Read all records from a JSONL file."""
    if not path.exists():
        return []
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    logger.warning("Corrupt JSONL line in %s, skipping", path.name)
    return records

print("Helpers loaded.")


Helpers loaded.


In [6]:
# ── Safe text normalization ──────────────────────────────────────
# Conservative: preserve legal structure, fix only whitespace/encoding issues.

def normalize_judgment_text(text: str) -> str:
    """Safe normalization for processing. Original text is preserved separately."""
    if is_missing(text):
        return ""
    t = str(text)
    # Unicode NFC normalization
    t = unicodedata.normalize("NFC", t)
    # Remove null bytes and other C0/C1 control chars (keep \n \t \r)
    t = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]", "", t)
    # Normalize various Unicode spaces to ASCII space
    t = re.sub(r"[\u00a0\u2000-\u200b\u202f\u205f\u3000\ufeff]", " ", t)
    # Collapse runs of spaces (not newlines) to single space
    t = re.sub(r"[^\S\n]+", " ", t)
    # Collapse 3+ consecutive blank lines to 2
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

# Quick sanity check
_test = "  हत्या   \x00\x0b  की   \n\n\n\n  धारा  "
assert "हत्या" in normalize_judgment_text(_test)
assert "\x00" not in normalize_judgment_text(_test)
assert "\n\n\n" not in normalize_judgment_text(_test)
print("Normalization function verified.")


Normalization function verified.


In [7]:
# ════════════════════════════════════════════════════════════════
# CASE-TYPE DEFINITIONS
# ════════════════════════════════════════════════════════════════

CRIMINAL_DEFINITIONS = """CASE-TYPE DEFINITIONS: CRIMINAL

SUBJECT: The accused person(s) alleged, prosecuted, tried, convicted, acquitted, or otherwise
substantively connected as the alleged actor of the underlying criminal event. Do NOT automatically
use procedural applicant/appellant labels without verifying the underlying criminal role.

OBJECT: The victim, injured person, deceased, or directly affected person/interest targeted or
harmed by the alleged criminal act. The informant/complainant is NOT automatically the Object
unless that person is also the actual victim or directly affected party.

OBJECTIVE ASPECT: The external criminal conduct — the legally relevant act or omission forming
the underlying criminal event (e.g. killing, assault, cruelty, theft, cheating, kidnapping,
sexual violence, criminal breach of trust, unlawful possession). Do NOT use procedural events
(filing bail, appeal, revision) as Objective Aspect when the judgment concerns an underlying
criminal event.

SUBJECTIVE ASPECT: The mental element — intention, knowledge, motive, recklessness, dishonest
intention, fraudulent intention, common intention, or other subjective component supported by the
judgment. Do NOT infer mens rea solely from a penal section number. If the judgment does not
clearly support the mental element, return: "Not Clearly Specified"
"""

CIVIL_DEFINITIONS = """CASE-TYPE DEFINITIONS: CIVIL

SUBJECT: The defendant, respondent, or substantive party against whom the civil claim, obligation,
liability, or enforceable demand is directed.

OBJECT: The plaintiff, claimant, decree-holder, or substantive party whose civil right or interest
is allegedly affected and who seeks civil relief.

OBJECTIVE ASPECT: The cause of action or legally relevant civil dispute — breach of contract,
property dispute, possession dispute, recovery claim, inheritance dispute, tortious injury, or
other civil wrong. Do NOT confuse procedural appeal/revision posture with the underlying civil dispute.

SUBJECTIVE ASPECT: The remedy sought or substantive civil relief pursued — declaration, injunction,
possession, recovery, damages, specific performance, partition, or other civil relief.
"""

CONSTITUTIONAL_DEFINITIONS = """CASE-TYPE DEFINITIONS: CONSTITUTIONAL

SUBJECT: The State, public authority, government body, or respondent authority whose action,
omission, law, decision, or conduct is challenged.

OBJECT: The petitioner or affected rights-holder alleging constitutional or public-law injury.

OBJECTIVE ASPECT: The alleged constitutional breach, rights violation, unlawful state action,
arbitrary action, jurisdictional error, detention issue, discrimination, due-process violation,
or other challenged public action. Do NOT use the filing of the writ petition itself as
Objective Aspect.

SUBJECTIVE ASPECT: The writ relief or constitutional remedy sought — habeas corpus, mandamus,
certiorari, prohibition, quo warranto, declaration, constitutional protection, or another
appropriate public-law remedy.
"""

ADMINISTRATIVE_DEFINITIONS = """CASE-TYPE DEFINITIONS: ADMINISTRATIVE

SUBJECT: The administrative authority, disciplinary authority, public body, department, tribunal,
or decision-making authority responsible for the impugned administrative action.

OBJECT: The aggrieved employee, applicant, license-holder, regulated person, beneficiary, or
other party directly affected by the administrative action.

OBJECTIVE ASPECT: The impugned administrative action — dismissal, suspension, transfer,
cancellation, blacklisting, disciplinary penalty, denial of benefit, recruitment decision,
licensing decision, or other administrative order/action. Do NOT confuse procedural filing with
the underlying impugned administrative action.

SUBJECTIVE ASPECT: The grounds for review or challenge — arbitrariness, procedural unfairness,
lack of jurisdiction, violation of natural justice, discrimination, illegality, proportionality,
or another supported review ground.
"""

CASE_DEFINITIONS = {
    "Criminal": CRIMINAL_DEFINITIONS,
    "Civil": CIVIL_DEFINITIONS,
    "Constitutional": CONSTITUTIONAL_DEFINITIONS,
    "Administrative": ADMINISTRATIVE_DEFINITIONS,
}

def get_case_definition(case_type: str) -> str:
    if case_type not in CASE_DEFINITIONS:
        raise ValueError("Unsupported case type: " + repr(case_type)
                         + ". Must be one of: " + ", ".join(CANONICAL_CASE_TYPES))
    return CASE_DEFINITIONS[case_type]

print("Case-type definitions loaded:", list(CASE_DEFINITIONS.keys()))


Case-type definitions loaded: ['Criminal', 'Civil', 'Constitutional', 'Administrative']


In [8]:
# ════════════════════════════════════════════════════════════════
# UNIVERSAL ANNOTATION PRINCIPLES & RULES
# ════════════════════════════════════════════════════════════════

UNIVERSAL_PRINCIPLES = """UNIVERSAL ANNOTATION PRINCIPLES

You are annotating Indian legal judgments for the INDILEX multilingual legal NLP dataset.

CORE RULES:
1. Identify the SUBSTANTIVE legal structure of the underlying dispute or event.
2. Do NOT confuse procedural role with substantive role.
3. Do NOT automatically equate: applicant = accused, appellant = accused,
   petitioner = victim, informant = victim, State = Object.
4. Do NOT treat appeal/revision/bail application as the underlying legal event.
5. Ground every annotation in the judgment text and extracted evidence.
6. If a value cannot be supported by evidence, use: "Not Clearly Specified"
7. Do NOT hallucinate facts not present in the judgment.
8. Do NOT use external legal knowledge to invent facts.
9. Distinguish allegation from court finding.
10. Distinguish role in proceeding from role in underlying event.
"""

LEGAL_PROVISION_RULES = """LEGAL PROVISION FIELD RULES

- Preserve section/article/rule number exactly as found.
- Preserve statute name when the judgment supports it.
- Do NOT map a provision to the wrong statute.
- Do NOT guess the statute from a bare section number.
- Distinguish provisions merely mentioned from provisions substantively applied, when evidence allows.
- Do NOT silently repair OCR-corrupted section numbers.
- If the section is visible but statute identity is unclear, preserve the uncertainty.
- Avoid hallucinated provisions.
- Use semicolons to separate multiple provisions.
- If no provisions are clearly identifiable, return empty string.
"""

REASONING_RULES = """REASONING FIELD RULES

Generate an evidence-grounded explanation of approximately 50-100 words.

The reasoning must justify the final annotation by connecting:
- Why the Subject has that substantive role
- Why the Object is the affected person/party/interest
- What underlying event or dispute supports the Objective Aspect
- What evidence supports the Subjective Aspect
- Which legal provisions are substantively relevant
- Relevant procedural posture where necessary
- Distinction between allegations, evidence, and court findings where relevant

Reasoning must NOT be: a generic case summary, a restatement of legal provisions only,
a list of sections, unsupported speculation, or invented facts.
"""

OUTPUT_RULES = """OUTPUT FORMAT (STRICT)

Return ONLY valid JSON with exactly these six keys:
{
  "subject": "",
  "object": "",
  "objective_aspect": "",
  "subjective_aspect": "",
  "legal_provision": "",
  "reasoning": ""
}

No Markdown. No code fences. No explanatory prefix or suffix. No additional keys.
"""

print("Universal principles and rules loaded.")


Universal principles and rules loaded.


In [9]:
# ════════════════════════════════════════════════════════════════
# STAGE A: CHUNK EVIDENCE EXTRACTION PROMPT
# ════════════════════════════════════════════════════════════════

EVIDENCE_EXTRACTION_SYSTEM = """You are an expert Indian legal analyst. Your task is to extract
STRUCTURED EVIDENCE from a chunk of a legal judgment.

CRITICAL INSTRUCTIONS:
- Extract ONLY evidence present in THIS chunk. Do NOT infer missing facts.
- Do NOT create final Subject/Object/Objective Aspect/Subjective Aspect labels yet.
- Do NOT treat applicant/appellant/petitioner as automatically the substantive legal actor.
- Do NOT treat informant as automatically the victim.
- SEPARATE role in proceeding from role in underlying event.
- SEPARATE allegation from court finding.
- Do NOT infer mens rea merely from a section number being mentioned.
- PRESERVE uncertainty — if something is unclear, mark it unclear.
- Do NOT repair uncertain OCR-corrupted section numbers by guessing.
- If a statute name is unclear, mark it "unclear" rather than guessing.
- Return ONLY valid JSON matching the schema below.

OUTPUT JSON SCHEMA:
{
  "chunk_id": <int>,
  "parties_and_roles": [
    {"name_or_description": "", "role_in_underlying_event": "", "role_in_proceeding": "", "evidence": ""}
  ],
  "underlying_events": [
    {"event": "", "actor": "", "affected_person_or_interest": "", "evidence": ""}
  ],
  "mental_state_evidence": [
    {"mental_state_or_motive": "", "evidence": "", "status": "allegation|evidence|court_finding|unclear"}
  ],
  "legal_provisions": [
    {"section": "", "statute": "", "context": "", "status": "charged|invoked|discussed|applied|convicted|acquitted|unclear"}
  ],
  "procedural_posture": [
    {"posture": "", "evidence": ""}
  ],
  "court_findings": [
    {"finding": "", "evidence": ""}
  ],
  "relief_or_outcome": [
    {"relief_or_outcome": "", "evidence": ""}
  ],
  "uncertainties": [
    {"issue": "", "reason": ""}
  ]
}

Return valid JSON ONLY. No Markdown, no code fences, no preamble, no suffix.
"""

def build_evidence_extraction_user_prompt(chunk_text: str, chunk_id: int,
                                          total_chunks: int, case_type: str,
                                          case_id: str) -> str:
    return (
        f"This is chunk {chunk_id + 1} of {total_chunks} from a {case_type} "
        f"legal judgment (case_id: {case_id}).\n\n"
        f"Extract structured evidence from this chunk ONLY.\n\n"
        f"<<<CHUNK_START>>>\n{chunk_text}\n<<<CHUNK_END>>>"
    )

print("Evidence extraction prompt template loaded.")


Evidence extraction prompt template loaded.


In [10]:
# ════════════════════════════════════════════════════════════════
# STAGE B: EVIDENCE AGGREGATION PROMPT
# ════════════════════════════════════════════════════════════════

AGGREGATION_SYSTEM = """You are an expert Indian legal analyst synthesizing evidence extracted
from multiple chunks of a SINGLE legal judgment.

INSTRUCTIONS:
- Combine evidence from all chunks into ONE consolidated evidence summary.
- Remove exact duplicates (same party mentioned identically in multiple chunks).
- Merge obviously repeated references to the same party, event, or provision.
- PRESERVE contradictory evidence (e.g. allegation in one chunk, acquittal in another).
- PRESERVE the distinction between allegation, evidence, and court finding.
- PRESERVE procedural-role vs underlying-event-role distinction.
- PRESERVE legal provision status (charged/invoked/discussed/applied/convicted/acquitted).
- PRESERVE uncertainty markers — do NOT resolve uncertainties by guessing.
- Do NOT silently discard conflicting evidence.
- Do NOT generate final Subject/Object/Objective Aspect/Subjective Aspect labels yet.
- Output the same JSON schema as the chunk evidence but consolidated.

Return valid JSON ONLY. No Markdown, no code fences, no preamble, no suffix.
"""

def build_aggregation_user_prompt(chunk_evidence_list: List[dict],
                                  case_type: str, case_id: str) -> str:
    return (
        f"Below is structured evidence from {len(chunk_evidence_list)} chunks of a "
        f"{case_type} legal judgment (case_id: {case_id}).\n\n"
        f"Synthesize into a single consolidated evidence summary.\n\n"
        f"CHUNK EVIDENCE:\n{json.dumps(chunk_evidence_list, ensure_ascii=False, indent=1)}"
    )

print("Evidence aggregation prompt template loaded.")


Evidence aggregation prompt template loaded.


In [11]:
# ════════════════════════════════════════════════════════════════
# STAGE C: CASE-CONDITIONED FINAL ANNOTATION PROMPT
# ════════════════════════════════════════════════════════════════

def build_final_annotation_system_prompt(case_type: str) -> str:
    """Build the system prompt with ONLY the relevant case-type definitions."""
    ct_def = get_case_definition(case_type)
    prompt = (
        UNIVERSAL_PRINCIPLES + "\n"
        + ct_def + "\n"
        + LEGAL_PROVISION_RULES + "\n"
        + REASONING_RULES + "\n"
        + OUTPUT_RULES
    )
    # Injection safety assertions
    marker = "CASE-TYPE DEFINITIONS: " + case_type.upper()
    assert marker in prompt, f"Own definition block missing for {case_type}"
    for other in CANONICAL_CASE_TYPES:
        if other != case_type:
            other_marker = "CASE-TYPE DEFINITIONS: " + other.upper()
            assert other_marker not in prompt, (
                f"CONTAMINATION: {other} definitions leaked into {case_type} prompt")
    return prompt

def build_final_annotation_user_prompt(aggregated_evidence: dict,
                                       case_type: str, case_id: str) -> str:
    return (
        f"Based on the following aggregated evidence from a {case_type} case "
        f"(case_id: {case_id}), generate the final six annotation fields.\n\n"
        f"AGGREGATED EVIDENCE:\n"
        f"{json.dumps(aggregated_evidence, ensure_ascii=False, indent=1)}\n\n"
        f"Return ONLY valid JSON with exactly these six keys: "
        f"subject, object, objective_aspect, subjective_aspect, legal_provision, reasoning."
    )

# Verify prompt isolation for all case types
for _ct in CANONICAL_CASE_TYPES:
    _ = build_final_annotation_system_prompt(_ct)
print("Final annotation prompt builder verified (all 4 case types isolated).")


Final annotation prompt builder verified (all 4 case types isolated).


In [12]:
# ════════════════════════════════════════════════════════════════
# JSON PARSING & REPAIR
# ════════════════════════════════════════════════════════════════

ANNOTATION_KEYS = {"subject", "object", "objective_aspect",
                   "subjective_aspect", "legal_provision", "reasoning"}

EVIDENCE_KEYS = {"parties_and_roles", "underlying_events", "mental_state_evidence",
                 "legal_provisions", "procedural_posture", "court_findings",
                 "relief_or_outcome", "uncertainties"}

def _try_loads(text: str) -> Optional[dict]:
    """Attempt multiple JSON parse strategies."""
    if not text or not text.strip():
        return None
    t = text.strip()
    # Strategy 1: direct parse
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        pass
    # Strategy 2: remove trailing commas before } or ]
    cleaned = re.sub(r",\s*([}\]])", r"\1", t)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    # Strategy 3: ast.literal_eval for Python-style dicts
    try:
        result = ast.literal_eval(t)
        if isinstance(result, dict):
            return result
    except (ValueError, SyntaxError):
        pass
    # Strategy 4: truncated JSON — try adding closing braces
    if t.count("{") > t.count("}"):
        patched = t + "}" * (t.count("{") - t.count("}"))
        patched = re.sub(r",\s*([}\]])", r"\1", patched)
        try:
            return json.loads(patched)
        except json.JSONDecodeError:
            pass
    return None

def robust_parse_json(raw_text: str, required_keys: set = None) -> Optional[dict]:
    """Parse model output, handling fences, prefix/suffix, malformation."""
    if raw_text is None or not str(raw_text).strip():
        return None
    text = str(raw_text).strip()

    # Remove markdown fences
    fence_match = re.search(r"```(?:json)?\s*\n?(.*?)```", text, re.DOTALL)
    if fence_match:
        text = fence_match.group(1).strip()

    # Try parsing the cleaned text
    result = _try_loads(text)

    # If that fails, try extracting JSON object from surrounding text
    if result is None:
        brace_match = re.search(r"\{.*\}", text, re.DOTALL)
        if brace_match:
            result = _try_loads(brace_match.group(0))

    if not isinstance(result, dict):
        return None

    # Check required keys if specified
    if required_keys:
        present = set(result.keys()) & required_keys
        if len(present) == 0:
            return None

    return result

def parse_evidence_output(raw_text: str) -> Optional[dict]:
    """Parse chunk evidence extraction output."""
    result = robust_parse_json(raw_text, EVIDENCE_KEYS)
    if result is None:
        return None
    # Normalize: ensure all evidence arrays exist, null -> []
    for key in EVIDENCE_KEYS:
        if key not in result or result[key] is None:
            result[key] = []
        elif not isinstance(result[key], list):
            result[key] = [result[key]] if result[key] else []
    return result

def parse_annotation_output(raw_text: str) -> Optional[dict]:
    """Parse final six-field annotation output."""
    result = robust_parse_json(raw_text, ANNOTATION_KEYS)
    if result is None:
        return None
    # Normalize: ensure all keys exist, null -> ""
    normalized = {}
    for key in ANNOTATION_KEYS:
        val = result.get(key)
        normalized[key] = "" if val is None else str(val).strip()
    return normalized

print("JSON parsing functions loaded.")


JSON parsing functions loaded.


## Phase 1 — Preprocessing (no model required)

**Steps:**
1. Load and validate input CSV
2. Split by case_type into batch CSVs (20 rows each)
3. Tokenize each judgment → compute token-aware chunk boundaries → save as JSONL

This phase uses only the tokenizer (lightweight), not the full model.
Run once; Phase 2 consumes the outputs.


In [ ]:
# ════════════════════════════════════════════════════════════════
# TOKENIZER (Phase 1 — no GPU required)
# ════════════════════════════════════════════════════════════════
from transformers import AutoTokenizer

TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH, trust_remote_code=True)
print("Tokenizer loaded:", MODEL_NAME_OR_PATH)
print("Vocab size:", TOKENIZER.vocab_size)


In [ ]:
# ════════════════════════════════════════════════════════════════
# TOKEN-AWARE CHUNKING
# ════════════════════════════════════════════════════════════════

def count_tokens(text: str) -> int:
    """Count tokens using the loaded tokenizer."""
    return len(TOKENIZER.encode(text, add_special_tokens=False))

def create_token_chunks(text: str,
                        chunk_size: int = CHUNK_SIZE_TOKENS,
                        overlap: int = CHUNK_OVERLAP_TOKENS
                       ) -> List[Dict[str, Any]]:
    """Split text into token-aware chunks with optional overlap.

    Returns list of dicts: {chunk_id, token_start, token_end, chunk_text}
    """
    token_ids = TOKENIZER.encode(text, add_special_tokens=False)
    total_tokens = len(token_ids)

    if total_tokens == 0:
        return []

    if total_tokens <= chunk_size:
        return [{
            "chunk_id": 0,
            "token_start": 0,
            "token_end": total_tokens,
            "chunk_text": TOKENIZER.decode(token_ids, skip_special_tokens=True),
        }]

    chunks = []
    step = max(1, chunk_size - overlap)
    start = 0
    chunk_id = 0
    while start < total_tokens:
        end = min(start + chunk_size, total_tokens)
        chunk_ids = token_ids[start:end]
        chunk_text = TOKENIZER.decode(chunk_ids, skip_special_tokens=True)
        chunks.append({
            "chunk_id": chunk_id,
            "token_start": start,
            "token_end": end,
            "chunk_text": chunk_text,
        })
        if end >= total_tokens:
            break
        start += step
        chunk_id += 1

    return chunks

# Quick verify
_short = "This is a short sentence."
_chunks = create_token_chunks(_short, chunk_size=9999)
assert len(_chunks) == 1 and _chunks[0]["chunk_id"] == 0
print("Chunking utilities loaded.")


In [ ]:
# ════════════════════════════════════════════════════════════════
# LOAD AND VALIDATE INPUT
# ════════════════════════════════════════════════════════════════

REQUIRED_COLUMNS = ["case_id", "language", "case_type", "judgment_text"]
TARGET_COLUMNS   = ["subject", "object", "objective_aspect",
                    "subjective_aspect", "legal_provision", "reasoning"]

# Try multiple encodings
for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
    try:
        INPUT_DF = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False, encoding=enc)
        logger.info("Input loaded with encoding=%s: %d rows", enc, len(INPUT_DF))
        break
    except (UnicodeDecodeError, UnicodeError):
        continue
else:
    raise RuntimeError("Cannot decode input CSV with any supported encoding")

# Validate required columns
missing_req = [c for c in REQUIRED_COLUMNS if c not in INPUT_DF.columns]
if missing_req:
    raise ValueError("Missing required columns: " + ", ".join(missing_req))

# Snapshot source columns for integrity checking later
SOURCE_COLUMNS = ["case_id", "language", "case_type", "judgment_text"]
SOURCE_SNAPSHOT = INPUT_DF[SOURCE_COLUMNS].copy()

# Normalize case_type
INPUT_DF["_original_case_type"] = INPUT_DF["case_type"]
INPUT_DF["case_type"] = INPUT_DF["case_type"].map(
    lambda v: normalize_case_type(v) or v)

# Detect unrecognized case types
known = set(CANONICAL_CASE_TYPES)
unknown_ct = INPUT_DF[~INPUT_DF["case_type"].isin(known)]
if len(unknown_ct) > 0:
    logger.warning("Unrecognized case types (%d rows): %s",
                   len(unknown_ct), unknown_ct["case_type"].unique().tolist())

# Summary
ct_counts = INPUT_DF["case_type"].value_counts().to_dict()
empty_jt = INPUT_DF["judgment_text"].map(is_missing).sum()
print("=" * 60)
print("INPUT VALIDATION SUMMARY")
print("=" * 60)
print("Total rows         :", len(INPUT_DF))
print("Case-type distribution:")
for ct, n in sorted(ct_counts.items()):
    print(f"  {ct:20s}: {n}")
print("Empty judgment_text:", empty_jt)
print("=" * 60)


In [ ]:
# ════════════════════════════════════════════════════════════════
# BATCH CREATION
# ════════════════════════════════════════════════════════════════

def create_batches(df: pd.DataFrame, batch_size: int = BATCH_SIZE
                  ) -> Tuple[List[dict], pd.DataFrame]:
    """Split DataFrame by case_type into batch CSVs of batch_size rows each.

    Returns (list of batch info dicts, manifest DataFrame).
    """
    manifest_rows = []
    all_batch_infos = []
    total_output_rows = 0

    for case_type in CANONICAL_CASE_TYPES:
        ct_df = df[df["case_type"] == case_type].copy()
        if len(ct_df) == 0:
            continue

        ct_lower = case_type.lower()
        ct_dir = BATCHES_DIR / case_type
        ct_dir.mkdir(parents=True, exist_ok=True)

        n_batches = (len(ct_df) + batch_size - 1) // batch_size
        for b_idx in range(n_batches):
            start = b_idx * batch_size
            end = min(start + batch_size, len(ct_df))
            batch_df = ct_df.iloc[start:end]

            batch_name = f"{ct_lower}_batch_{b_idx + 1:03d}"
            batch_path = ct_dir / f"{batch_name}.csv"
            atomic_save_csv(batch_df, batch_path)

            info = {
                "batch_name": batch_name,
                "case_type": case_type,
                "batch_path": str(batch_path),
                "row_count": len(batch_df),
                "first_case_id": batch_df.iloc[0]["case_id"],
                "last_case_id": batch_df.iloc[-1]["case_id"],
            }
            manifest_rows.append(info)
            all_batch_infos.append(info)
            total_output_rows += len(batch_df)

            logger.info("Batch %s: %d rows", batch_name, len(batch_df))

    # Validate: no rows lost or gained
    valid_rows = len(df[df["case_type"].isin(CANONICAL_CASE_TYPES)])
    assert total_output_rows == valid_rows, (
        f"Batch row count mismatch: {total_output_rows} vs {valid_rows} valid input rows")

    manifest_df = pd.DataFrame(manifest_rows)
    manifest_path = BATCHES_DIR / "batch_manifest.csv"
    atomic_save_csv(manifest_df, manifest_path)
    logger.info("Manifest saved: %s (%d batches)", manifest_path, len(manifest_df))

    return all_batch_infos, manifest_df


def create_chunk_files(batch_infos: List[dict]) -> dict:
    """Tokenize all judgments in each batch and save chunk JSONL files.

    Returns dict of batch_name -> chunk_stats.
    """
    stats = {}
    for info in tqdm(batch_infos, desc="Chunking batches"):
        batch_name = info["batch_name"]
        case_type = info["case_type"]
        batch_df = pd.read_csv(info["batch_path"], dtype=str,
                               keep_default_na=False, encoding="utf-8-sig")

        ct_chunk_dir = CHUNKS_DIR / case_type
        ct_chunk_dir.mkdir(parents=True, exist_ok=True)
        chunk_path = ct_chunk_dir / f"{batch_name}_chunks.jsonl"

        total_chunks = 0
        token_counts = []

        with open(chunk_path, "w", encoding="utf-8") as f:
            for _, row in batch_df.iterrows():
                case_id = row["case_id"]
                jtext = safe_str(row["judgment_text"])
                if is_missing(jtext):
                    # Empty judgment: write zero-chunk record
                    rec = {"case_id": case_id, "total_chunks": 0,
                           "chunks": [], "total_tokens": 0, "status": "empty"}
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    token_counts.append(0)
                    continue

                normalized = normalize_judgment_text(jtext)
                n_tokens = count_tokens(normalized)
                chunks = create_token_chunks(normalized,
                                             CHUNK_SIZE_TOKENS,
                                             CHUNK_OVERLAP_TOKENS)
                chunk_records = []
                for ch in chunks:
                    chunk_records.append({
                        "chunk_id": ch["chunk_id"],
                        "token_start": ch["token_start"],
                        "token_end": ch["token_end"],
                        "chunk_text": ch["chunk_text"],
                    })

                rec = {"case_id": case_id, "total_chunks": len(chunks),
                       "chunks": chunk_records, "total_tokens": n_tokens,
                       "status": "chunked"}
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                total_chunks += len(chunks)
                token_counts.append(n_tokens)

        stats[batch_name] = {
            "total_chunks": total_chunks,
            "total_judgments": len(batch_df),
            "avg_tokens": sum(token_counts) / max(len(token_counts), 1),
            "max_tokens": max(token_counts) if token_counts else 0,
            "chunk_path": str(chunk_path),
        }
        logger.info("Chunks for %s: %d judgments -> %d chunks (avg %.0f tokens)",
                    batch_name, len(batch_df), total_chunks,
                    stats[batch_name]["avg_tokens"])

    return stats

print("Batch creation and chunking functions loaded.")


In [ ]:
# ════════════════════════════════════════════════════════════════
# EXECUTE PHASE 1: Batching + Chunking
# ════════════════════════════════════════════════════════════════

MANIFEST_PATH = BATCHES_DIR / "batch_manifest.csv"

if MANIFEST_PATH.exists() and not FORCE_REPROCESS_PHASE1:
    print("Phase 1 output already exists. Loading manifest from previous run.")
    print("Set FORCE_REPROCESS_PHASE1 = True to re-create batches and chunks.")
    BATCH_MANIFEST = pd.read_csv(MANIFEST_PATH, dtype=str, keep_default_na=False,
                                 encoding="utf-8-sig")
    BATCH_INFOS = BATCH_MANIFEST.to_dict("records")
    # Verify chunk files exist
    for info in BATCH_INFOS:
        ct = info["case_type"]
        bn = info["batch_name"]
        cp = CHUNKS_DIR / ct / f"{bn}_chunks.jsonl"
        assert cp.exists(), f"Chunk file missing: {cp}"
    print("Loaded", len(BATCH_INFOS), "batches from manifest.")
else:
    print("Running Phase 1: creating batches and computing chunks...")
    BATCH_INFOS, BATCH_MANIFEST = create_batches(INPUT_DF, BATCH_SIZE)
    CHUNK_STATS = create_chunk_files(BATCH_INFOS)

    # Print summary
    print("\n" + "=" * 60)
    print("PHASE 1 COMPLETE")
    print("=" * 60)
    print("Batches created     :", len(BATCH_INFOS))
    for ct in CANONICAL_CASE_TYPES:
        ct_batches = [b for b in BATCH_INFOS if b["case_type"] == ct]
        if ct_batches:
            total_rows = sum(int(b["row_count"]) for b in ct_batches)
            print(f"  {ct:20s}: {len(ct_batches)} batches, {total_rows} rows")

    total_chunks = sum(s["total_chunks"] for s in CHUNK_STATS.values())
    total_j = sum(s["total_judgments"] for s in CHUNK_STATS.values())
    print("Total chunks        :", total_chunks)
    print("Avg chunks/judgment :", f"{total_chunks / max(total_j, 1):.1f}")
    print("=" * 60)


## Phase 2 — Annotation (model required, resumable)

**Per judgment (independent):**
1. Load pre-computed chunks from JSONL
2. **Stage A:** Extract structured evidence from each chunk (N calls)
3. **Stage B:** Aggregate cross-chunk evidence (1 call)
4. **Stage C:** Generate case-conditioned final annotation (1 call)
5. Validate JSON → checkpoint → next judgment

Model calls per judgment: **N + 2** (where N = number of chunks).

Checkpointed at every level: chunk evidence, aggregation, final annotation.
Resume skips all completed work.


In [13]:
!watch -n 1 nvidia-smi

>---------------------------------------+----------------------+-------------7715 2026889952020112322263314455775882399263030111225334429156507768819954040811622235332644166877588992350502111228

In [14]:
# ════════════════════════════════════════════════════════════════
# MODEL LOADING
# ════════════════════════════════════════════════════════════════
import torch

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count       :", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        mem = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  GPU {i}: {name} ({mem:.1f} GB)")
else:
    print("WARNING: No CUDA GPU detected. Large model inference will be extremely slow on CPU.")

MODEL = None
API_CALL_COUNT = 0

if DRY_RUN:
    print("\nDRY_RUN = True -> model loading SKIPPED (no model calls will be made).")
else:
    if BACKEND == "vllm":
        from vllm import LLM, SamplingParams

        print(f"\nLoading model via vLLM: {MODEL_NAME_OR_PATH}")
        print(f"  tensor_parallel_size = {TENSOR_PARALLEL_SIZE}")
        print(f"  gpu_memory_utilization = {GPU_MEMORY_UTILIZATION}")

        MODEL = LLM(
            model=MODEL_NAME_OR_PATH,
            tensor_parallel_size=TENSOR_PARALLEL_SIZE,
            gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
            dtype="auto",
            trust_remote_code=True,
            max_model_len=None,  # auto-detect from model config
            disable_custom_all_reduce=True,
        )
        print("Model loaded successfully via vLLM.")
    elif BACKEND == "transformers":
        from transformers import AutoModelForCausalLM
        print(f"\nLoading model via transformers: {MODEL_NAME_OR_PATH}")
        MODEL = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME_OR_PATH,
            torch_dtype="auto",
            device_map="auto",
            trust_remote_code=True,
        )
        MODEL.eval()
        print("Model loaded successfully via transformers.")
    else:
        raise ValueError(f"Unknown BACKEND: {BACKEND}. Use 'vllm' or 'transformers'.")


PyTorch version : 2.4.0+cu121
CUDA available  : True
GPU count       : 2
  GPU 0: NVIDIA GeForce RTX 3090 (23.7 GB)
  GPU 1: NVIDIA GeForce RTX 3090 (23.7 GB)


2026-07-08 19:58:59,623	INFO util.py:154 -- Outdated packages:
  ipywidgets==7.6.5 found, needs ipywidgets>=8
Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.



Loading model via vLLM: Qwen/Qwen2.5-7B-Instruct
  tensor_parallel_size = 2
  gpu_memory_utilization = 0.9
INFO 07-08 19:59:01 config.py:904] Defaulting to use mp for distributed inference
INFO 07-08 19:59:01 llm_engine.py:223] Initializing an LLM engine (v0.6.1.post1) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=2, pipeline_parallel_size=1, disable_custom_all_reduce=True, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_ti

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 07-08 20:00:13 model_runner.py:1008] Loading model weights took 7.1216 GB
(VllmWorkerProcess pid=2434) INFO 07-08 20:00:14 model_runner.py:1008] Loading model weights took 7.1216 GB
INFO 07-08 20:00:22 distributed_gpu_executor.py:57] # GPU blocks: 25123, # CPU blocks: 9362
INFO 07-08 20:00:26 model_runner.py:1309] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 07-08 20:00:26 model_runner.py:1313] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
(VllmWorkerProcess pid=2434) INFO 07-08 20:00:27 model_runner.py:1309] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager

In [ ]:
!watch -n 1 nvidia-smi

In [15]:
# ════════════════════════════════════════════════════════════════
# GENERATE TEXT — backend-agnostic abstraction
# ════════════════════════════════════════════════════════════════

def generate_text(system_prompt: str, user_prompt: str,
                  max_new_tokens: int = MAX_NEW_TOKENS_FINAL) -> str:
    """Generate text using the loaded model. Backend-agnostic."""
    global API_CALL_COUNT

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    formatted = TOKENIZER.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)

    if BACKEND == "vllm":
        from vllm import SamplingParams
        params = SamplingParams(
            temperature=TEMPERATURE,
            max_tokens=max_new_tokens,
            top_p=1.0,
        )
        outputs = MODEL.generate([formatted], params, use_tqdm=False)
        result = outputs[0].outputs[0].text.strip()

    elif BACKEND == "transformers":
        input_ids = TOKENIZER(formatted, return_tensors="pt").input_ids
        input_ids = input_ids.to(MODEL.device)
        with torch.no_grad():
            output_ids = MODEL.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                pad_token_id=TOKENIZER.pad_token_id or TOKENIZER.eos_token_id,
            )
        new_tokens = output_ids[0][input_ids.shape[1]:]
        result = TOKENIZER.decode(new_tokens, skip_special_tokens=True).strip()
    else:
        raise ValueError(f"Unknown BACKEND: {BACKEND}")

    API_CALL_COUNT += 1
    if DEBUG:
        print(f"[DEBUG generate_text] call #{API_CALL_COUNT}, "
              f"output length: {len(result)} chars")
    return result

print("generate_text() abstraction ready | backend:", BACKEND)


generate_text() abstraction ready | backend: vllm


In [16]:
# ════════════════════════════════════════════════════════════════
# CHECKPOINT MANAGER
# ════════════════════════════════════════════════════════════════

CHECKPOINT_COLUMNS = [
    "case_id", "batch_name", "case_type",
    "chunk_count", "completed_chunks",
    "evidence_status",      # pending / complete / failed
    "aggregation_status",   # pending / complete / failed / skipped
    "annotation_status",    # pending / success / failed / skipped_empty
    "retry_count", "error_message",
    "subject", "object", "objective_aspect",
    "subjective_aspect", "legal_provision", "reasoning",
    "model_name", "timestamp",
]

def load_checkpoint(batch_name: str) -> pd.DataFrame:
    """Load checkpoint CSV for a batch, or return empty DataFrame."""
    path = CHECKPOINTS_DIR / f"{batch_name}_checkpoint.csv"
    if path.exists():
        try:
            df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
            logger.info("Checkpoint loaded: %s (%d rows)", path.name, len(df))
            return df
        except Exception as e:
            logger.warning("Corrupt checkpoint %s: %s — starting fresh", path.name, e)
    return pd.DataFrame(columns=CHECKPOINT_COLUMNS)

def save_checkpoint(batch_name: str, records: List[dict]) -> None:
    """Save checkpoint CSV atomically."""
    if DRY_RUN:
        return
    df = pd.DataFrame(records)
    for col in CHECKPOINT_COLUMNS:
        if col not in df.columns:
            df[col] = ""
    path = CHECKPOINTS_DIR / f"{batch_name}_checkpoint.csv"
    atomic_save_csv(df[CHECKPOINT_COLUMNS], path)
    logger.debug("Checkpoint saved: %s (%d rows)", path.name, len(df))

def get_completed_chunk_ids(batch_name: str, case_id: str) -> set:
    """Read evidence JSONL and return set of completed chunk_ids for a case."""
    ct = None
    for info in BATCH_INFOS:
        if info["batch_name"] == batch_name:
            ct = info["case_type"]
            break
    if ct is None:
        return set()
    evidence_path = CHUNK_EVIDENCE_DIR / ct / f"{batch_name}_evidence.jsonl"
    completed = set()
    for rec in read_jsonl(evidence_path):
        if rec.get("case_id") == case_id and rec.get("status") == "success":
            completed.add(rec.get("chunk_id"))
    return completed

def load_chunk_evidence(batch_name: str, case_id: str, case_type: str) -> List[dict]:
    """Load all successfully extracted chunk evidence for a case."""
    evidence_path = CHUNK_EVIDENCE_DIR / case_type / f"{batch_name}_evidence.jsonl"
    results = []
    for rec in read_jsonl(evidence_path):
        if rec.get("case_id") == case_id and rec.get("status") == "success":
            results.append(rec.get("evidence", {}))
    return sorted(results, key=lambda e: e.get("chunk_id", 0))

def load_aggregated_evidence(batch_name: str, case_id: str, case_type: str) -> Optional[dict]:
    """Load aggregated evidence for a case, if it exists."""
    agg_path = AGGREGATED_EVIDENCE_DIR / case_type / f"{batch_name}_aggregated.jsonl"
    for rec in read_jsonl(agg_path):
        if rec.get("case_id") == case_id and rec.get("status") == "success":
            return rec.get("aggregated_evidence", {})
    return None

print("Checkpoint manager loaded.")


Checkpoint manager loaded.


In [17]:
# ════════════════════════════════════════════════════════════════
# STAGE A: CHUNK EVIDENCE EXTRACTION
# ════════════════════════════════════════════════════════════════

def extract_chunk_evidence(chunk_text: str, chunk_id: int, total_chunks: int,
                           case_type: str, case_id: str,
                           batch_name: str) -> Tuple[Optional[dict], str]:
    """Extract structured evidence from one chunk.

    Returns (evidence_dict or None, error_message).
    """
    user_prompt = build_evidence_extraction_user_prompt(
        chunk_text, chunk_id, total_chunks, case_type, case_id)

    last_error = ""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            raw = generate_text(EVIDENCE_EXTRACTION_SYSTEM, user_prompt,
                                max_new_tokens=MAX_NEW_TOKENS_EVIDENCE)
            evidence = parse_evidence_output(raw)
            if evidence is not None:
                evidence["chunk_id"] = chunk_id
                return evidence, ""
            last_error = f"JSON parse failed on attempt {attempt}"
            logger.warning("case_id=%s chunk=%d attempt %d/%d: parse failed",
                          case_id, chunk_id, attempt, MAX_RETRIES)
        except torch.cuda.OutOfMemoryError:
            last_error = "CUDA OOM"
            logger.error("case_id=%s chunk=%d: CUDA OOM", case_id, chunk_id)
            torch.cuda.empty_cache()
            gc.collect()
        except Exception as e:
            last_error = f"{type(e).__name__}: {e}"
            logger.warning("case_id=%s chunk=%d attempt %d/%d: %s",
                          case_id, chunk_id, attempt, MAX_RETRIES, last_error)

    return None, last_error

print("Chunk evidence extraction function loaded.")


Chunk evidence extraction function loaded.


In [18]:
# ════════════════════════════════════════════════════════════════
# EVIDENCE PRE-PROCESSING (deterministic dedup before model aggregation)
# ════════════════════════════════════════════════════════════════

def dedup_evidence_entries(entries: list) -> list:
    """Remove exact-duplicate dicts from a list of evidence entries."""
    seen = set()
    result = []
    for entry in entries:
        key = json.dumps(entry, sort_keys=True, ensure_ascii=False)
        if key not in seen:
            seen.add(key)
            result.append(entry)
    return result

def python_dedup_evidence(chunk_evidence_list: List[dict]) -> dict:
    """Merge all chunk evidence arrays, removing exact duplicates.

    This is the deterministic pre-processing step before model-based synthesis.
    """
    merged = {}
    for key in EVIDENCE_KEYS:
        combined = []
        for chunk_ev in chunk_evidence_list:
            combined.extend(chunk_ev.get(key, []))
        merged[key] = dedup_evidence_entries(combined)
    return merged

print("Evidence dedup functions loaded.")


Evidence dedup functions loaded.


In [19]:
# ════════════════════════════════════════════════════════════════
# STAGE B: EVIDENCE AGGREGATION (Python dedup + model synthesis)
# ════════════════════════════════════════════════════════════════

def aggregate_evidence(chunk_evidence_list: List[dict],
                       case_type: str, case_id: str,
                       batch_name: str) -> Tuple[Optional[dict], str]:
    """Aggregate evidence from all chunks of one judgment.

    Step 1: Python-level deterministic dedup (free, instant).
    Step 2: Model-based synthesis to merge near-duplicates and reconcile contradictions.

    For single-chunk judgments, skip model call — the chunk evidence IS the aggregation.

    Returns (aggregated_evidence or None, error_message).
    """
    if len(chunk_evidence_list) == 0:
        return None, "No chunk evidence to aggregate"

    # Single chunk: no aggregation needed
    if len(chunk_evidence_list) == 1:
        return chunk_evidence_list[0], ""

    # Step 1: Python dedup
    deduped = python_dedup_evidence(chunk_evidence_list)

    # Step 2: Model synthesis
    user_prompt = build_aggregation_user_prompt(
        chunk_evidence_list, case_type, case_id)

    # Check if evidence fits in context (rough estimate)
    prompt_tokens = count_tokens(AGGREGATION_SYSTEM + user_prompt)
    logger.info("case_id=%s aggregation prompt: ~%d tokens", case_id, prompt_tokens)

    last_error = ""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            raw = generate_text(AGGREGATION_SYSTEM, user_prompt,
                                max_new_tokens=MAX_NEW_TOKENS_AGGREGATION)
            aggregated = parse_evidence_output(raw)
            if aggregated is not None:
                return aggregated, ""
            last_error = f"JSON parse failed on attempt {attempt}"
            logger.warning("case_id=%s aggregation attempt %d/%d: parse failed",
                          case_id, attempt, MAX_RETRIES)
        except torch.cuda.OutOfMemoryError:
            last_error = "CUDA OOM during aggregation"
            logger.error("case_id=%s aggregation: CUDA OOM", case_id)
            torch.cuda.empty_cache()
            gc.collect()
            # Fallback: use Python-deduped evidence without model synthesis
            if attempt == MAX_RETRIES:
                logger.warning("case_id=%s: falling back to Python-only dedup", case_id)
                return deduped, "OOM_fallback_to_python_dedup"
        except Exception as e:
            last_error = f"{type(e).__name__}: {e}"
            logger.warning("case_id=%s aggregation attempt %d/%d: %s",
                          case_id, attempt, MAX_RETRIES, last_error)

    # Final fallback: return Python-deduped evidence
    logger.warning("case_id=%s: model aggregation failed, using Python dedup", case_id)
    return deduped, "fallback_to_python_dedup: " + last_error

print("Evidence aggregation function loaded.")


Evidence aggregation function loaded.


In [20]:
# ════════════════════════════════════════════════════════════════
# STAGE C: FINAL CASE-CONDITIONED ANNOTATION
# ════════════════════════════════════════════════════════════════

def generate_final_annotation(aggregated_evidence: dict,
                              case_type: str, case_id: str
                             ) -> Tuple[Optional[dict], str]:
    """Generate the final six annotation fields from aggregated evidence.

    Returns (annotation_dict or None, error_message).
    """
    system_prompt = build_final_annotation_system_prompt(case_type)
    user_prompt = build_final_annotation_user_prompt(
        aggregated_evidence, case_type, case_id)

    last_error = ""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            raw = generate_text(system_prompt, user_prompt,
                                max_new_tokens=MAX_NEW_TOKENS_FINAL)
            annotation = parse_annotation_output(raw)
            if annotation is not None:
                # Semantic check: reject all-empty annotations
                values = [annotation[k] for k in ANNOTATION_KEYS if k != "reasoning"]
                if all(is_missing(v) for v in values):
                    last_error = f"Semantically empty annotation on attempt {attempt}"
                    logger.warning("case_id=%s final attempt %d/%d: all fields empty",
                                  case_id, attempt, MAX_RETRIES)
                    continue
                return annotation, ""
            last_error = f"JSON parse failed on attempt {attempt}"
            logger.warning("case_id=%s final attempt %d/%d: parse failed",
                          case_id, attempt, MAX_RETRIES)
        except torch.cuda.OutOfMemoryError:
            last_error = "CUDA OOM during final annotation"
            logger.error("case_id=%s final annotation: CUDA OOM", case_id)
            torch.cuda.empty_cache()
            gc.collect()
        except Exception as e:
            last_error = f"{type(e).__name__}: {e}"
            logger.warning("case_id=%s final attempt %d/%d: %s",
                          case_id, attempt, MAX_RETRIES, last_error)

    return None, last_error

print("Final annotation function loaded.")


Final annotation function loaded.


In [21]:
# ════════════════════════════════════════════════════════════════
# PER-JUDGMENT PIPELINE (orchestrates stages A → B → C)
# ════════════════════════════════════════════════════════════════

def process_single_judgment(case_id: str, case_type: str,
                            chunks_data: dict, batch_name: str,
                            checkpoint_rec: dict) -> dict:
    """Process one judgment through evidence → aggregation → annotation.

    Updates and returns the checkpoint record.
    """
    total_chunks = chunks_data.get("total_chunks", 0)
    chunk_list = chunks_data.get("chunks", [])
    checkpoint_rec["chunk_count"] = str(total_chunks)

    # Handle empty judgment
    if total_chunks == 0 or chunks_data.get("status") == "empty":
        checkpoint_rec["annotation_status"] = "skipped_empty"
        checkpoint_rec["evidence_status"] = "skipped"
        checkpoint_rec["aggregation_status"] = "skipped"
        checkpoint_rec["timestamp"] = now_iso()
        logger.info("case_id=%s: skipped (empty judgment)", case_id)
        return checkpoint_rec

    ct_evidence_dir = CHUNK_EVIDENCE_DIR / case_type
    ct_evidence_dir.mkdir(parents=True, exist_ok=True)
    evidence_path = ct_evidence_dir / f"{batch_name}_evidence.jsonl"

    ct_agg_dir = AGGREGATED_EVIDENCE_DIR / case_type
    ct_agg_dir.mkdir(parents=True, exist_ok=True)
    agg_path = ct_agg_dir / f"{batch_name}_aggregated.jsonl"

    # ── STAGE A: Chunk evidence extraction ──────────────────────
    if checkpoint_rec.get("evidence_status") != "complete":
        completed_ids = get_completed_chunk_ids(batch_name, case_id)
        checkpoint_rec["completed_chunks"] = str(len(completed_ids))

        for chunk_data in chunk_list:
            cid = chunk_data["chunk_id"]
            if cid in completed_ids:
                continue  # already done — skip

            evidence, error = extract_chunk_evidence(
                chunk_data["chunk_text"], cid, total_chunks,
                case_type, case_id, batch_name)

            if evidence is not None:
                append_jsonl({
                    "case_id": case_id, "chunk_id": cid,
                    "evidence": evidence, "status": "success",
                }, evidence_path)
                completed_ids.add(cid)
                checkpoint_rec["completed_chunks"] = str(len(completed_ids))
            else:
                append_jsonl({
                    "case_id": case_id, "chunk_id": cid,
                    "evidence": None, "status": "failed", "error": error,
                }, evidence_path)
                logger.error("case_id=%s chunk=%d: evidence extraction failed: %s",
                            case_id, cid, error)

        # Check if all chunks succeeded
        completed_ids = get_completed_chunk_ids(batch_name, case_id)
        if len(completed_ids) >= total_chunks:
            checkpoint_rec["evidence_status"] = "complete"
        else:
            checkpoint_rec["evidence_status"] = "partial"
            # Still proceed with available evidence

    # ── STAGE B: Evidence aggregation ───────────────────────────
    if checkpoint_rec.get("aggregation_status") != "complete":
        chunk_evidence = load_chunk_evidence(batch_name, case_id, case_type)

        if len(chunk_evidence) == 0:
            checkpoint_rec["aggregation_status"] = "failed"
            checkpoint_rec["annotation_status"] = "failed"
            checkpoint_rec["error_message"] = "No chunk evidence available for aggregation"
            checkpoint_rec["timestamp"] = now_iso()
            return checkpoint_rec

        aggregated, agg_error = aggregate_evidence(
            chunk_evidence, case_type, case_id, batch_name)

        if aggregated is not None:
            append_jsonl({
                "case_id": case_id,
                "aggregated_evidence": aggregated,
                "status": "success",
                "note": agg_error if agg_error else "",
            }, agg_path)
            checkpoint_rec["aggregation_status"] = "complete"
        else:
            checkpoint_rec["aggregation_status"] = "failed"
            checkpoint_rec["annotation_status"] = "failed"
            checkpoint_rec["error_message"] = "Aggregation failed: " + agg_error
            checkpoint_rec["timestamp"] = now_iso()
            return checkpoint_rec

    # ── STAGE C: Final annotation ──────────────────────────────
    if checkpoint_rec.get("annotation_status") not in ("success",):
        aggregated = load_aggregated_evidence(batch_name, case_id, case_type)
        if aggregated is None:
            checkpoint_rec["annotation_status"] = "failed"
            checkpoint_rec["error_message"] = "Aggregated evidence not found"
            checkpoint_rec["timestamp"] = now_iso()
            return checkpoint_rec

        annotation, ann_error = generate_final_annotation(
            aggregated, case_type, case_id)

        if annotation is not None:
            for key in ANNOTATION_KEYS:
                checkpoint_rec[key] = annotation.get(key, "")
            checkpoint_rec["annotation_status"] = "success"
            checkpoint_rec["error_message"] = ""
        else:
            checkpoint_rec["annotation_status"] = "failed"
            checkpoint_rec["error_message"] = "Final annotation failed: " + ann_error

    checkpoint_rec["model_name"] = MODEL_NAME_OR_PATH
    checkpoint_rec["timestamp"] = now_iso()
    return checkpoint_rec

print("Per-judgment pipeline loaded.")


Per-judgment pipeline loaded.


In [22]:
# ════════════════════════════════════════════════════════════════
# MAIN PROCESSING LOOP
# ════════════════════════════════════════════════════════════════

RUN_START_TIME = time.time()
RUN_STATS = defaultdict(int)

def dry_run_demo(batch_infos: List[dict]) -> None:
    """DRY_RUN: show prompts and chunking without any model calls."""
    print("=" * 60)
    print("DRY RUN — verifying pipeline without model calls")
    print("=" * 60)

    # Pick first non-empty judgment
    for info in batch_infos:
        ct = info["case_type"]
        bn = info["batch_name"]
        chunk_path = CHUNKS_DIR / ct / f"{bn}_chunks.jsonl"
        for rec in read_jsonl(chunk_path):
            if rec.get("total_chunks", 0) > 0:
                cid = rec["case_id"]
                chunks = rec["chunks"]
                print(f"\nSelected: case_id={cid}, case_type={ct}, batch={bn}")
                print(f"Total tokens: {rec['total_tokens']}, Chunks: {rec['total_chunks']}")

                # Show chunk previews
                for ch in chunks[:3]:
                    preview = ch["chunk_text"][:200] + "..." if len(ch["chunk_text"]) > 200 else ch["chunk_text"]
                    print(f"\n  Chunk {ch['chunk_id']}: tokens {ch['token_start']}-{ch['token_end']}")
                    print(f"  Preview: {preview}")
                if len(chunks) > 3:
                    print(f"  ... ({len(chunks) - 3} more chunks)")

                # Show evidence extraction prompt
                print("\n" + "-" * 40)
                print("EVIDENCE EXTRACTION SYSTEM PROMPT (first 500 chars):")
                print(EVIDENCE_EXTRACTION_SYSTEM[:500])
                user_p = build_evidence_extraction_user_prompt(
                    chunks[0]["chunk_text"][:500] + "...", 0, len(chunks), ct, cid)
                print("\nEVIDENCE EXTRACTION USER PROMPT (truncated):")
                print(user_p[:500])

                # Show aggregation prompt structure
                print("\n" + "-" * 40)
                print("AGGREGATION SYSTEM PROMPT (first 300 chars):")
                print(AGGREGATION_SYSTEM[:300])

                # Show final annotation prompt + case-type isolation
                print("\n" + "-" * 40)
                sys_prompt = build_final_annotation_system_prompt(ct)
                print(f"FINAL ANNOTATION SYSTEM PROMPT for {ct} (first 500 chars):")
                print(sys_prompt[:500])

                # Verify case-type isolation
                print("\n" + "-" * 40)
                print("CASE-TYPE DEFINITION ISOLATION CHECK:")
                for check_ct in CANONICAL_CASE_TYPES:
                    sp = build_final_annotation_system_prompt(check_ct)
                    own_marker = f"CASE-TYPE DEFINITIONS: {check_ct.upper()}"
                    present = own_marker in sp
                    others = []
                    for o in CANONICAL_CASE_TYPES:
                        if o != check_ct:
                            om = f"CASE-TYPE DEFINITIONS: {o.upper()}"
                            if om in sp:
                                others.append(o)
                    status = "PASS" if (present and not others) else "FAIL"
                    print(f"  {check_ct:20s}: own={present}, leaked={others or 'none'} -> {status}")

                print("\n" + "=" * 60)
                print("DRY RUN COMPLETE — no model calls made, no data modified.")
                print("=" * 60)
                return

    print("No non-empty judgments found for dry-run demo.")


def process_all_batches(batch_infos: List[dict]) -> None:
    """Process all batches (or selected ones) through the annotation pipeline."""
    global API_CALL_COUNT

    # Filter batches if PROCESS_BATCHES is set
    if PROCESS_BATCHES is not None:
        batch_infos = [b for b in batch_infos if b["batch_name"] in PROCESS_BATCHES]
        print(f"Processing selected batches: {[b['batch_name'] for b in batch_infos]}")

    for info in batch_infos:
        batch_name = info["batch_name"]
        case_type = info["case_type"]
        batch_path = info["batch_path"]

        print(f"\n{'=' * 60}")
        print(f"Processing batch: {batch_name} ({case_type})")
        print(f"{'=' * 60}")

        # Load batch data
        batch_df = pd.read_csv(batch_path, dtype=str,
                               keep_default_na=False, encoding="utf-8-sig")

        # Load pre-computed chunks
        chunk_path = CHUNKS_DIR / case_type / f"{batch_name}_chunks.jsonl"
        chunk_records = read_jsonl(chunk_path)
        chunks_by_case = {r["case_id"]: r for r in chunk_records}

        # Load existing checkpoint
        ckpt_df = load_checkpoint(batch_name)
        ckpt_by_case = {}
        for _, row in ckpt_df.iterrows():
            ckpt_by_case[row["case_id"]] = row.to_dict()

        # Build processing list
        checkpoint_records = []
        judgments_processed = 0

        pbar = tqdm(batch_df.iterrows(), total=len(batch_df),
                    desc=batch_name, unit="judgment")
        try:
            for idx, row in pbar:
                case_id = row["case_id"]

                # Initialize or restore checkpoint record
                if case_id in ckpt_by_case:
                    rec = ckpt_by_case[case_id]
                else:
                    rec = {col: "" for col in CHECKPOINT_COLUMNS}
                    rec["case_id"] = case_id
                    rec["batch_name"] = batch_name
                    rec["case_type"] = case_type

                # Skip already successful
                if rec.get("annotation_status") == "success":
                    checkpoint_records.append(rec)
                    RUN_STATS["skipped_complete"] += 1
                    continue

                # Skip already failed unless retry is enabled
                if rec.get("annotation_status") == "failed" and not RETRY_FAILED_ON_RESUME:
                    checkpoint_records.append(rec)
                    RUN_STATS["skipped_failed"] += 1
                    continue

                # Reset failed status for retry
                if rec.get("annotation_status") == "failed" and RETRY_FAILED_ON_RESUME:
                    # Keep evidence/aggregation status if complete
                    if rec.get("annotation_status") == "failed":
                        rec["annotation_status"] = "pending"
                        rec["error_message"] = ""

                # MAX_JUDGMENTS limit
                if MAX_JUDGMENTS is not None and judgments_processed >= MAX_JUDGMENTS:
                    rec["annotation_status"] = rec.get("annotation_status", "pending")
                    checkpoint_records.append(rec)
                    continue

                # Get chunk data
                chunks_data = chunks_by_case.get(case_id, {"total_chunks": 0, "chunks": [], "status": "empty"})

                # Process judgment
                try:
                    rec = process_single_judgment(
                        case_id, case_type, chunks_data, batch_name, rec)

                    if rec["annotation_status"] == "success":
                        RUN_STATS["success"] += 1
                    elif rec["annotation_status"] == "skipped_empty":
                        RUN_STATS["skipped_empty"] += 1
                    elif rec["annotation_status"] == "failed":
                        RUN_STATS["failed"] += 1

                except KeyboardInterrupt:
                    raise
                except Exception as e:
                    rec["annotation_status"] = "failed"
                    rec["error_message"] = f"{type(e).__name__}: {e}"
                    rec["timestamp"] = now_iso()
                    RUN_STATS["failed"] += 1
                    logger.error("case_id=%s: unhandled error: %s\n%s",
                                case_id, e, traceback.format_exc())

                checkpoint_records.append(rec)
                judgments_processed += 1
                RUN_STATS["attempted"] += 1

                # Periodic checkpoint save
                if judgments_processed % 5 == 0:
                    save_checkpoint(batch_name, checkpoint_records)

        except KeyboardInterrupt:
            print("\nInterrupted! Saving checkpoint...")
            save_checkpoint(batch_name, checkpoint_records)
            print("Checkpoint saved. Resume will continue from here.")
            raise
        finally:
            pbar.close()

        # Final checkpoint save for this batch
        save_checkpoint(batch_name, checkpoint_records)
        s_count = sum(1 for r in checkpoint_records if r.get("annotation_status") == "success")
        f_count = sum(1 for r in checkpoint_records if r.get("annotation_status") == "failed")
        print(f"Batch {batch_name}: {s_count} success, {f_count} failed, "
              f"{len(checkpoint_records)} total")


# ── Execute ─────────────────────────────────────────────────────
if DRY_RUN:
    dry_run_demo(BATCH_INFOS)
else:
    process_all_batches(BATCH_INFOS)


NameError: name 'BATCH_INFOS' is not defined

In [ ]:
# ════════════════════════════════════════════════════════════════
# FINAL DATAFRAME CONSTRUCTION
# ════════════════════════════════════════════════════════════════

FINAL_COLUMNS = ["case_id", "language", "case_type", "judgment_text",
                 "subject", "object", "objective_aspect",
                 "subjective_aspect", "legal_provision", "reasoning"]

def build_final_dataframe(input_df: pd.DataFrame,
                          batch_infos: List[dict]) -> pd.DataFrame:
    """Merge checkpoint annotations back into the original DataFrame."""
    # Start with source columns from input
    final = input_df[SOURCE_COLUMNS].copy()

    # Initialize annotation columns
    for col in TARGET_COLUMNS:
        final[col] = ""

    # Merge annotations from checkpoints
    for info in batch_infos:
        batch_name = info["batch_name"]
        ckpt_path = CHECKPOINTS_DIR / f"{batch_name}_checkpoint.csv"
        if not ckpt_path.exists():
            continue

        ckpt = pd.read_csv(ckpt_path, dtype=str, keep_default_na=False,
                           encoding="utf-8-sig")

        for _, row in ckpt.iterrows():
            if row.get("annotation_status") != "success":
                continue
            case_id = row["case_id"]
            mask = final["case_id"] == case_id
            if mask.sum() == 0:
                logger.warning("case_id=%s from checkpoint not found in input", case_id)
                continue
            if mask.sum() > 1:
                # Duplicate case_id: match by position within batch
                logger.warning("case_id=%s has %d matches; using first unassigned",
                              case_id, mask.sum())
                unassigned = mask & final["subject"].eq("")
                if unassigned.sum() > 0:
                    idx = final[unassigned].index[0]
                else:
                    idx = final[mask].index[0]
            else:
                idx = final[mask].index[0]

            for col in TARGET_COLUMNS:
                val = row.get(col, "")
                final.at[idx, col] = val if not is_missing(val) else ""

    return final[FINAL_COLUMNS]

if not DRY_RUN:
    FINAL_DF = build_final_dataframe(INPUT_DF, BATCH_INFOS)
    print("Final DataFrame constructed:", FINAL_DF.shape)
    print("Columns:", list(FINAL_DF.columns))
else:
    print("DRY_RUN: final DataFrame not constructed.")
    FINAL_DF = None


In [ ]:
# ════════════════════════════════════════════════════════════════
# INTEGRITY VALIDATION
# ════════════════════════════════════════════════════════════════

if not DRY_RUN and FINAL_DF is not None:
    # Check schema
    assert list(FINAL_DF.columns) == FINAL_COLUMNS, (
        "Column order mismatch: " + str(list(FINAL_DF.columns)))

    # Check row count
    assert len(FINAL_DF) == len(INPUT_DF), (
        f"Row count mismatch: final={len(FINAL_DF)} vs input={len(INPUT_DF)}")

    # Check source column integrity
    for col in SOURCE_COLUMNS:
        final_vals = FINAL_DF[col].reset_index(drop=True)
        source_vals = SOURCE_SNAPSHOT[col].reset_index(drop=True)
        if not final_vals.equals(source_vals):
            raise RuntimeError(
                f"SOURCE COLUMN INTEGRITY VIOLATION: '{col}' was modified. "
                f"Final output NOT saved. Checkpoint data preserved.")

    print("Integrity validation PASSED:")
    print("  Column order  : correct")
    print("  Row count     : correct (" + str(len(FINAL_DF)) + ")")
    print("  Source columns: byte-identical to input")
else:
    print("DRY_RUN: integrity validation skipped.")


In [ ]:
# ════════════════════════════════════════════════════════════════
# SAVE FINAL CLEAN CSV
# ════════════════════════════════════════════════════════════════

FINAL_OUTPUT_PATH = FINAL_DIR / "INDILEX_long_annotated.csv"

if not DRY_RUN and FINAL_DF is not None:
    atomic_save_csv(FINAL_DF, FINAL_OUTPUT_PATH)
    logger.info("Final CSV saved: %s (%d rows)", FINAL_OUTPUT_PATH, len(FINAL_DF))
    print("Final annotated CSV saved:", FINAL_OUTPUT_PATH)
    print("Rows:", len(FINAL_DF), "| Columns:", len(FINAL_DF.columns))
else:
    print("DRY_RUN: final CSV not saved.")


In [ ]:
# ════════════════════════════════════════════════════════════════
# ERROR REPORT
# ════════════════════════════════════════════════════════════════

if not DRY_RUN:
    all_errors = []
    for info in BATCH_INFOS:
        ckpt_path = CHECKPOINTS_DIR / f"{info['batch_name']}_checkpoint.csv"
        if not ckpt_path.exists():
            continue
        ckpt = pd.read_csv(ckpt_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
        failed = ckpt[ckpt["annotation_status"].isin(["failed", "skipped_empty"])]
        if len(failed) > 0:
            all_errors.append(failed[["case_id", "batch_name", "case_type",
                                      "annotation_status", "error_message",
                                      "retry_count", "timestamp"]])

    if all_errors:
        ERROR_REPORT = pd.concat(all_errors, ignore_index=True)
        error_path = ERRORS_DIR / "error_report.csv"
        atomic_save_csv(ERROR_REPORT, error_path)
        print(f"Error report saved: {error_path} ({len(ERROR_REPORT)} rows)")
    else:
        ERROR_REPORT = pd.DataFrame()
        print("No errors to report.")
else:
    print("DRY_RUN: error report skipped.")


In [ ]:
# ════════════════════════════════════════════════════════════════
# RUN SUMMARY
# ════════════════════════════════════════════════════════════════

elapsed = time.time() - RUN_START_TIME

print("=" * 60)
print("COMPLETION REPORT")
print("=" * 60)
print("Input file            :", INPUT_PATH)
print("Output directory      :", OUTPUT_DIR)
print("Model                 :", MODEL_NAME_OR_PATH)
print("Backend               :", BACKEND)
print("DRY_RUN               :", DRY_RUN)
print("-" * 60)

if not DRY_RUN:
    # Gather stats from checkpoints
    total_success = 0
    total_failed = 0
    total_empty = 0
    total_pending = 0
    total_chunks_processed = 0

    for info in BATCH_INFOS:
        ckpt_path = CHECKPOINTS_DIR / f"{info['batch_name']}_checkpoint.csv"
        if ckpt_path.exists():
            ckpt = pd.read_csv(ckpt_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
            statuses = ckpt["annotation_status"].value_counts().to_dict()
            total_success += statuses.get("success", 0)
            total_failed += statuses.get("failed", 0)
            total_empty += statuses.get("skipped_empty", 0)
            total_pending += statuses.get("pending", 0) + statuses.get("", 0)

    print("Total rows            :", len(INPUT_DF))
    print("Batches               :", len(BATCH_INFOS))
    print("Successful annotations:", total_success)
    print("Failed                :", total_failed)
    print("Skipped (empty)       :", total_empty)
    print("Pending               :", total_pending)
    print("Total API calls       :", API_CALL_COUNT)
    print("Elapsed time          :", f"{elapsed:.1f} seconds")
    print("-" * 60)

    if FINAL_DF is not None:
        print("Per-field missing counts:")
        for f in TARGET_COLUMNS:
            n_missing = int(FINAL_DF[f].map(is_missing).sum())
            print(f"  {f:20s}: {n_missing}")

    print("-" * 60)
    print("Output files:")
    print("  Final CSV     :", FINAL_OUTPUT_PATH if FINAL_OUTPUT_PATH.exists() else "(not saved)")
    print("  Manifest      :", BATCHES_DIR / "batch_manifest.csv")
    print("  Checkpoints   :", CHECKPOINTS_DIR)
    print("  Chunk evidence:", CHUNK_EVIDENCE_DIR)
    print("  Aggregated    :", AGGREGATED_EVIDENCE_DIR)
    print("  Errors        :", ERRORS_DIR)
    print("  Logs          :", LOG_PATH)
else:
    print("DRY_RUN complete. No model calls made. No data modified.")
    print("Elapsed time:", f"{elapsed:.1f} seconds")

print("=" * 60)
logger.info("RUN COMPLETE | api_calls=%d elapsed=%.1fs", API_CALL_COUNT, elapsed)


## Self-Tests (no model calls)

The cell below tests helper functions, chunking, prompt routing, and JSON parsing
without making any model calls. Run it any time to verify pipeline plumbing.


In [ ]:
# ════════════════════════════════════════════════════════════════
# SELF-TESTS (no model calls)
# ════════════════════════════════════════════════════════════════

def run_self_tests():
    # ── 1. Missing judgment detection ───────────────────────────
    assert is_missing(None) and is_missing(float("nan")) and is_missing("")
    assert is_missing("   ") and is_missing("nan") and is_missing("NaN")
    assert is_missing("none") and is_missing("NULL") and is_missing("n/a")
    assert not is_missing("Ram Kumar") and not is_missing("0")
    print("  [1] is_missing: PASS")

    # ── 2. Whitespace-only judgment ─────────────────────────────
    assert is_missing("   \n\t  ")
    assert normalize_judgment_text("   \n\t  ") == ""
    print("  [2] whitespace-only: PASS")

    # ── 3. Unicode normalization ────────────────────────────────
    t = normalize_judgment_text("\u00a0हत्या\x00  की\n\n\n\n  धारा")
    assert "\x00" not in t and "\u00a0" not in t
    assert "हत्या" in t and "धारा" in t
    assert "\n\n\n" not in t
    print("  [3] Unicode normalization: PASS")

    # ── 4-5. Token chunk creation + overlap ─────────────────────
    # Use a simple test (tokenizer must be loaded)
    short_text = "One two three four five."
    chunks = create_token_chunks(short_text, chunk_size=9999)
    assert len(chunks) == 1 and chunks[0]["chunk_id"] == 0
    assert chunks[0]["token_start"] == 0

    # Create a longer text that forces multiple chunks
    long_text = "word " * 500
    token_count = count_tokens(long_text)
    if token_count > 20:
        chunks_small = create_token_chunks(long_text, chunk_size=10, overlap=2)
        assert len(chunks_small) > 1, "Should create multiple chunks"
        assert chunks_small[0]["chunk_id"] == 0
        assert chunks_small[1]["chunk_id"] == 1
        # Verify overlap: second chunk starts before first chunk ends
        if len(chunks_small) >= 2:
            step = 10 - 2
            assert chunks_small[1]["token_start"] == step
    print("  [4] chunk creation: PASS")
    print("  [5] chunk overlap: PASS")

    # ── 6-7. Batch size behavior ────────────────────────────────
    test_rows = [{"case_id": f"T{i:03d}", "language": "hi",
                  "case_type": "Criminal", "judgment_text": f"text {i}"}
                 for i in range(25)]
    test_df = pd.DataFrame(test_rows)

    import tempfile
    with tempfile.TemporaryDirectory() as tmpdir:
        old_batches = BATCHES_DIR
        old_chunks = CHUNKS_DIR
        # Temporarily redirect
        globals()["BATCHES_DIR"] = Path(tmpdir) / "batches"
        globals()["CHUNKS_DIR"] = Path(tmpdir) / "chunks"
        BATCHES_DIR.mkdir(parents=True)
        CHUNKS_DIR.mkdir(parents=True)

        infos, manifest = create_batches(test_df, batch_size=20)
        assert len(infos) == 2  # 20 + 5
        assert int(infos[0]["row_count"]) == 20
        assert int(infos[1]["row_count"]) == 5  # final smaller batch
        total = sum(int(b["row_count"]) for b in infos)
        assert total == 25

        # Restore
        globals()["BATCHES_DIR"] = old_batches
        globals()["CHUNKS_DIR"] = old_chunks
    print("  [6] batch size 20: PASS")
    print("  [7] final smaller batch: PASS")

    # ── 8. Case-type prompt routing ─────────────────────────────
    for ct in CANONICAL_CASE_TYPES:
        sp = build_final_annotation_system_prompt(ct)
        own_marker = f"CASE-TYPE DEFINITIONS: {ct.upper()}"
        assert own_marker in sp, f"Missing own definition for {ct}"
        assert "UNIVERSAL ANNOTATION PRINCIPLES" in sp
        assert "LEGAL PROVISION FIELD" in sp
        assert "REASONING FIELD" in sp
        assert "OUTPUT FORMAT (STRICT)" in sp
        for other in CANONICAL_CASE_TYPES:
            if other != ct:
                other_marker = f"CASE-TYPE DEFINITIONS: {other.upper()}"
                assert other_marker not in sp, f"{other} leaked into {ct} prompt"
    try:
        build_final_annotation_system_prompt("Maritime")
        raise AssertionError("Should reject unsupported case type")
    except ValueError:
        pass
    print("  [8] case-type routing isolation: PASS")

    # ── 9-16. JSON parsing ──────────────────────────────────────
    base = {"subject": "A", "object": "B", "objective_aspect": "C",
            "subjective_aspect": "D", "legal_provision": "Sec 302 IPC",
            "reasoning": "R"}
    clean = json.dumps(base)

    # 9: clean JSON
    assert parse_annotation_output(clean) == base
    print("  [9] clean JSON: PASS")

    # 10: fenced JSON
    assert parse_annotation_output("```json\n" + clean + "\n```") == base
    print("  [10] fenced JSON: PASS")

    # 11: prefix + JSON
    assert parse_annotation_output("Here is the annotation:\n" + clean) == base
    print("  [11] prefix + JSON: PASS")

    # 12: JSON + suffix
    assert parse_annotation_output(clean + "\nHope this helps!") == base
    print("  [12] JSON + suffix: PASS")

    # 13: null values
    with_null = json.dumps({**base, "subjective_aspect": None})
    r = parse_annotation_output(with_null)
    assert r["subjective_aspect"] == ""
    print("  [13] null values: PASS")

    # 14: missing keys
    partial = json.dumps({"subject": "A", "reasoning": "R"})
    r = parse_annotation_output(partial)
    assert r["subject"] == "A" and r["object"] == ""
    print("  [14] missing keys: PASS")

    # 15: extra keys
    extra = json.dumps({**base, "confidence": 0.9})
    assert parse_annotation_output(extra) == base
    print("  [15] extra keys: PASS")

    # 16: malformed JSON
    assert parse_annotation_output("I cannot annotate this.") is None
    assert parse_annotation_output("") is None
    assert parse_annotation_output(None) is None
    print("  [16] malformed JSON: PASS")

    # Extra: single quotes
    assert parse_annotation_output(str(base)) == base
    print("  [16b] single-quote dict: PASS")

    # Extra: trailing comma
    assert parse_annotation_output(clean[:-1] + ",}") == base
    print("  [16c] trailing comma: PASS")

    # ── Evidence JSON parsing ───────────────────────────────────
    ev_base = {
        "chunk_id": 0,
        "parties_and_roles": [{"name_or_description": "Ramesh", "role_in_underlying_event": "accused"}],
        "underlying_events": [],
        "mental_state_evidence": [],
        "legal_provisions": [{"section": "302", "statute": "IPC"}],
        "procedural_posture": [],
        "court_findings": [],
        "relief_or_outcome": [],
        "uncertainties": [],
    }
    r = parse_evidence_output(json.dumps(ev_base))
    assert r is not None and len(r["parties_and_roles"]) == 1
    print("  [16d] evidence JSON parse: PASS")

    # ── 17. Duplicate case_id detection ─────────────────────────
    dup_df = pd.DataFrame([
        {"case_id": "X1", "language": "hi", "case_type": "Criminal", "judgment_text": "t1"},
        {"case_id": "X1", "language": "hi", "case_type": "Criminal", "judgment_text": "t2"},
    ])
    assert dup_df["case_id"].duplicated().sum() == 1
    print("  [17] duplicate case_id detection: PASS")

    # ── 18. Source-column integrity ─────────────────────────────
    test_source = pd.DataFrame({"case_id": ["A"], "language": ["hi"],
                                "case_type": ["Criminal"], "judgment_text": ["text"]})
    snapshot = test_source.copy()
    assert test_source.equals(snapshot)
    test_source.at[0, "case_id"] = "CHANGED"
    assert not test_source["case_id"].equals(snapshot["case_id"])
    print("  [18] source integrity check: PASS")

    # ── 19. Checkpoint resume simulation ────────────────────────
    import tempfile
    with tempfile.TemporaryDirectory() as tmpdir:
        old_ckpt = CHECKPOINTS_DIR
        old_dry = globals().get("DRY_RUN", False)
        globals()["CHECKPOINTS_DIR"] = Path(tmpdir)
        globals()["DRY_RUN"] = False  # override for test
        recs = [{"case_id": "T1", "batch_name": "test_batch_001", "case_type": "Criminal",
                 "annotation_status": "success", "subject": "S1"}]
        save_checkpoint("test_batch_001", recs)
        loaded = load_checkpoint("test_batch_001")
        assert len(loaded) == 1 and loaded.iloc[0]["case_id"] == "T1"
        assert loaded.iloc[0]["annotation_status"] == "success"
        globals()["CHECKPOINTS_DIR"] = old_ckpt
        globals()["DRY_RUN"] = old_dry
    print("  [19] checkpoint resume: PASS")

    # ── 20. Completed-chunk skip ────────────────────────────────
    # Verify get_completed_chunk_ids returns correct set
    import tempfile
    with tempfile.TemporaryDirectory() as tmpdir:
        old_ev = CHUNK_EVIDENCE_DIR
        globals()["CHUNK_EVIDENCE_DIR"] = Path(tmpdir)
        ct_dir = Path(tmpdir) / "Criminal"
        ct_dir.mkdir()
        ev_path = ct_dir / "test_batch_001_evidence.jsonl"
        append_jsonl({"case_id": "T1", "chunk_id": 0, "status": "success", "evidence": {}}, ev_path)
        append_jsonl({"case_id": "T1", "chunk_id": 1, "status": "success", "evidence": {}}, ev_path)
        append_jsonl({"case_id": "T1", "chunk_id": 2, "status": "failed", "evidence": None}, ev_path)

        # Mock BATCH_INFOS temporarily
        old_bi = BATCH_INFOS
        globals()["BATCH_INFOS"] = [{"batch_name": "test_batch_001", "case_type": "Criminal"}]
        completed = get_completed_chunk_ids("test_batch_001", "T1")
        assert completed == {0, 1}, f"Expected {{0,1}}, got {completed}"
        globals()["BATCH_INFOS"] = old_bi
        globals()["CHUNK_EVIDENCE_DIR"] = old_ev
    print("  [20] completed-chunk skip: PASS")

    # ── Case-type normalization ─────────────────────────────────
    assert normalize_case_type("Criminal") == "Criminal"
    assert normalize_case_type(" CRIMINAL ") == "Criminal"
    assert normalize_case_type("civil") == "Civil"
    assert normalize_case_type("constitutional") == "Constitutional"
    assert normalize_case_type("ADMIN") == "Administrative"
    assert normalize_case_type("Maritime") is None
    assert normalize_case_type("") is None
    print("  [+] case-type normalization: PASS")

    # ── Evidence dedup ──────────────────────────────────────────
    entries = [{"name": "A"}, {"name": "B"}, {"name": "A"}]
    assert len(dedup_evidence_entries(entries)) == 2
    print("  [+] evidence dedup: PASS")

    print("\n  ALL SELF-TESTS PASSED")

run_self_tests()
